# Data Preparation


## Setup

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import os
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

BASE_PATH = "/content/drive/MyDrive/Reflow"
PROCESSED_PATH = f"{BASE_PATH}/data/processed"

TRAIN_DAILY_PATH = f"{PROCESSED_PATH}/train_daily_enriched.parquet"
EVAL_DAILY_PATH = f"{PROCESSED_PATH}/eval_daily_enriched.parquet"

TRAIN_HOURLY_PATH = f"{PROCESSED_PATH}/train_hourly.parquet"
EVAL_HOURLY_PATH = f"{PROCESSED_PATH}/eval_hourly.parquet"

print("Input  train:", TRAIN_DAILY_PATH)
print("Input  eval :", EVAL_DAILY_PATH)
print("Output train:", TRAIN_HOURLY_PATH)
print("Output eval :", EVAL_HOURLY_PATH)


Input  train: /content/drive/MyDrive/Reflow/data/processed/train_daily_enriched.parquet
Input  eval : /content/drive/MyDrive/Reflow/data/processed/eval_daily_enriched.parquet
Output train: /content/drive/MyDrive/Reflow/data/processed/train_hourly.parquet
Output eval : /content/drive/MyDrive/Reflow/data/processed/eval_hourly.parquet


In [4]:
train_df = pd.read_parquet(TRAIN_DAILY_PATH)
eval_df = pd.read_parquet(EVAL_DAILY_PATH)

train_df["dt"] = pd.to_datetime(train_df["dt"])
eval_df["dt"] = pd.to_datetime(eval_df["dt"])

print("Train (daily):", train_df.shape)
print("Eval  (daily):", eval_df.shape)


Train (daily): (4500000, 19)
Eval  (daily): (350000, 19)


In [5]:
sales_matrix = np.stack(train_df["hours_sale"].values)
stock_matrix = np.stack(train_df["hours_stock_status"].values)

sales_when_stockout = sales_matrix[stock_matrix == 1]
sales_when_available = sales_matrix[stock_matrix == 0]

pct_nonzero_during_stockout = (sales_when_stockout > 0).mean() * 100

print(f"Rata-rata sales SAAT stockout        : {sales_when_stockout.mean():.5f}")
print(f"Rata-rata sales SAAT stok tersedia    : {sales_when_available.mean():.5f}")
print(f"% observasi stockout dengan sales > 0 : {pct_nonzero_during_stockout:.2f}%")


Rata-rata sales SAAT stockout        : 0.00420
Rata-rata sales SAAT stok tersedia    : 0.05400
% observasi stockout dengan sales > 0 : 2.87%


## Fungsi transformasi: explode harian -> hourly (vectorized)


In [6]:
STATIC_COLUMNS = [
    "store_id", "product_id", "city_id", "management_group_id",
    "first_category_id", "second_category_id", "third_category_id",
    "discount", "holiday_flag", "activity_flag",
    "precpt", "avg_temperature", "avg_humidity", "avg_wind_level",
]

def explode_chunk(df_chunk):
    """Ubah 1 chunk dataframe harian (1 baris = 1 hari, array 24 jam)
    menjadi dataframe hourly (1 baris = 1 jam)."""
    n = len(df_chunk)

    hour = np.tile(np.arange(24, dtype=np.int8), n)
    dt_repeated = np.repeat(df_chunk["dt"].values, 24)
    datetime_arr = dt_repeated + hour.astype("timedelta64[h]")
    weekday = np.repeat(df_chunk["dt"].dt.dayofweek.values.astype(np.int8), 24)

    sales = np.concatenate(df_chunk["hours_sale"].values).astype(np.float32)
    stock = np.concatenate(df_chunk["hours_stock_status"].values).astype(np.int8)

    latent_demand = np.where(stock == 1, np.nan, sales).astype(np.float32)

    out = {
        "datetime": datetime_arr,
        "hour": hour,
        "weekday": weekday,
        "observed_sales": sales,
        "is_stockout": stock,
        "latent_demand": latent_demand,
    }

    for col in STATIC_COLUMNS:
        dtype = df_chunk[col].dtype
        out[col] = np.repeat(df_chunk[col].values, 24)

    return pd.DataFrame(out)


### Sanity check fungsi transformasi pada 1 baris



In [7]:
check_row = train_df.sample(1, random_state=7)
exploded_check = explode_chunk(check_row)

print("Baris asli:")
print(check_row[["store_id", "product_id", "dt"] + ["hours_sale", "hours_stock_status"]].T)

print("\nHasil explode (24 baris):")
print(exploded_check[["datetime", "hour", "observed_sales", "is_stockout", "latent_demand"]])

original_sales = check_row["hours_sale"].values[0]
original_stock = check_row["hours_stock_status"].values[0]

assert np.allclose(exploded_check["observed_sales"].values, original_sales), "Sales tidak cocok!"
assert np.array_equal(exploded_check["is_stockout"].values, original_stock), "Stock status tidak cocok!"
print("\nValidasi OK: hasil explode identik dengan array asli.")


Baris asli:
                                                              3195318
store_id                                                          602
product_id                                                        715
dt                                                2024-05-15 00:00:00
hours_sale          [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.2, ...
hours_stock_status  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...

Hasil explode (24 baris):
              datetime  hour  observed_sales  is_stockout  latent_demand
0  2024-05-15 00:00:00     0            0.00            0           0.00
1  2024-05-15 01:00:00     1            0.00            0           0.00
2  2024-05-15 02:00:00     2            0.00            0           0.00
3  2024-05-15 03:00:00     3            0.00            0           0.00
4  2024-05-15 04:00:00     4            0.00            0           0.00
5  2024-05-15 05:00:00     5            0.00            0           0.00
6  2024-05-15 06:00:00     6  

In [8]:
def transform_to_hourly(df, output_path, chunk_size=100_000):
    writer = None
    n_chunks = (len(df) + chunk_size - 1) // chunk_size
    total_rows_written = 0

    for i, start in enumerate(range(0, len(df), chunk_size)):
        chunk = df.iloc[start:start + chunk_size]
        exploded = explode_chunk(chunk)

        table = pa.Table.from_pandas(exploded, preserve_index=False)

        if writer is None:
            writer = pq.ParquetWriter(output_path, table.schema, compression="snappy")

        writer.write_table(table)
        total_rows_written += len(exploded)

        if (i + 1) % 5 == 0 or (i + 1) == n_chunks:
            print(f"  chunk {i + 1}/{n_chunks} -> total baris ditulis: {total_rows_written:,}")

    if writer is not None:
        writer.close()

    print(f"Selesai. Total baris hourly: {total_rows_written:,} (harusnya = {len(df):,} x 24 = {len(df) * 24:,})")
    return total_rows_written


In [9]:
print("=== Transform TRAIN ===")
train_rows_written = transform_to_hourly(train_df, TRAIN_HOURLY_PATH, chunk_size=100_000)


=== Transform TRAIN ===
  chunk 5/45 -> total baris ditulis: 12,000,000
  chunk 10/45 -> total baris ditulis: 24,000,000
  chunk 15/45 -> total baris ditulis: 36,000,000
  chunk 20/45 -> total baris ditulis: 48,000,000
  chunk 25/45 -> total baris ditulis: 60,000,000
  chunk 30/45 -> total baris ditulis: 72,000,000
  chunk 35/45 -> total baris ditulis: 84,000,000
  chunk 40/45 -> total baris ditulis: 96,000,000
  chunk 45/45 -> total baris ditulis: 108,000,000
Selesai. Total baris hourly: 108,000,000 (harusnya = 4,500,000 x 24 = 108,000,000)


In [10]:
print("=== Transform EVAL ===")
eval_rows_written = transform_to_hourly(eval_df, EVAL_HOURLY_PATH, chunk_size=100_000)


=== Transform EVAL ===
  chunk 4/4 -> total baris ditulis: 8,400,000
Selesai. Total baris hourly: 8,400,000 (harusnya = 350,000 x 24 = 8,400,000)


## Validasi hasil akhir



In [11]:
!ls -lh "{TRAIN_HOURLY_PATH}" "{EVAL_HOURLY_PATH}"


-rw------- 1 root root 8.3M Aug 19 05:08 /content/drive/MyDrive/Reflow/data/processed/eval_hourly.parquet
-rw------- 1 root root 112M Aug 19 05:02 /content/drive/MyDrive/Reflow/data/processed/train_hourly.parquet


In [12]:
train_hourly_meta = pq.ParquetFile(TRAIN_HOURLY_PATH).metadata
eval_hourly_meta = pq.ParquetFile(EVAL_HOURLY_PATH).metadata

print("TRAIN hourly rows:", train_hourly_meta.num_rows, "  (expected:", len(train_df) * 24, ")")
print("EVAL  hourly rows:", eval_hourly_meta.num_rows, "  (expected:", len(eval_df) * 24, ")")

assert train_hourly_meta.num_rows == len(train_df) * 24
assert eval_hourly_meta.num_rows == len(eval_df) * 24
print("\nValidasi jumlah baris: OK")


TRAIN hourly rows: 108000000   (expected: 108000000 )
EVAL  hourly rows: 8400000   (expected: 8400000 )

Validasi jumlah baris: OK


In [13]:
sample_hourly = pd.read_parquet(TRAIN_HOURLY_PATH, columns=["is_stockout", "latent_demand", "observed_sales"])

censored_pct = sample_hourly["is_stockout"].mean() * 100
null_latent_pct = sample_hourly["latent_demand"].isnull().mean() * 100

print(f"% baris is_stockout=1        : {censored_pct:.2f}%  (dari EDA: sekitar 24.9%)")
print(f"% baris latent_demand = NaN  : {null_latent_pct:.2f}%  (harus PERSIS SAMA dengan angka di atas)")

assert abs(censored_pct - null_latent_pct) < 1e-6, "latent_demand NaN tidak cocok dengan is_stockout!"
print("\nValidasi definisi target: OK")


% baris is_stockout=1        : 24.89%  (dari EDA: sekitar 24.9%)
% baris latent_demand = NaN  : 24.89%  (harus PERSIS SAMA dengan angka di atas)

Validasi definisi target: OK


In [14]:
print(pq.ParquetFile(TRAIN_HOURLY_PATH).schema)


required group field_id=-1 schema {
  optional int64 field_id=-1 datetime (Timestamp(isAdjustedToUTC=false, timeUnit=nanoseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional int32 field_id=-1 hour (Int(bitWidth=8, isSigned=true));
  optional int32 field_id=-1 weekday (Int(bitWidth=8, isSigned=true));
  optional float field_id=-1 observed_sales;
  optional int32 field_id=-1 is_stockout (Int(bitWidth=8, isSigned=true));
  optional float field_id=-1 latent_demand;
  optional int64 field_id=-1 store_id;
  optional int64 field_id=-1 product_id;
  optional int64 field_id=-1 city_id;
  optional int64 field_id=-1 management_group_id;
  optional int64 field_id=-1 first_category_id;
  optional int64 field_id=-1 second_category_id;
  optional int64 field_id=-1 third_category_id;
  optional double field_id=-1 discount;
  optional int32 field_id=-1 holiday_flag;
  optional int32 field_id=-1 activity_flag;
  optional double field_id=-1 precpt;
  optional double field_i